# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")

# Cargamos los datos preprocesados
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
n_features = 561;

# Modelos básicos y selección de atributos (20%)

In [ ]:
# Definición de diccionarios de configuración
dic_filtros = Dict(
    "Sin_Filtrado" => nothing,
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [2]:
function run_experiment(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                              X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # --- 1. PREPARACIÓN DE RESULTADOS ---
    if isfile(output_file)
        println("Archivo de checkpoint encontrado: $output_file")
        results_df = CSV.read(output_file, DataFrame)
        # Creamos un set de identificadores únicos para saltar lo ya hecho
        combinaciones_hechas = Set([
            (string(r.Filter), string(r.Reduction), string(r.Model)) 
            for r in eachrow(results_df)
        ])
        println("$(length(combinaciones_hechas)) experimentos completados previamente.")
    else
        println("Iniciando experimento desde cero.")
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy_List = String[], 
            Accuracy_Mean = Float64[], F1_Score = Float64[], B_Accuracy = Float64[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # --- CORRECCIÓN DE MÉTRICAS ---
    # Usamos balanced_accuracy en lugar de recall para evitar errores multiclase
    measures = [accuracy, multiclass_f1score, balanced_accuracy]

    # --- 2. BUCLE PRINCIPAL ---
    for filt_name in sort(collect(keys(dic_filtros)))
        filt_model = dic_filtros[filt_name]
        
        for red_name in sort(collect(keys(dic_reducciones)))
            red_model = dic_reducciones[red_name]
            
            for mod_name in sort(collect(keys(dic_modelos)))
                mod_model = dic_modelos[mod_name]
                
                # Si ya existe esta combinación, saltamos
                if (filt_name, red_name, mod_name) in combinaciones_hechas
                    continue 
                end

                println("\nEvaluando: [$filt_name] + [$red_name] + [$mod_name]")
                
                # 1. Definimos el Pipeline
                pipe = PersonalizedPipeline(
                    scaler    = MyMinMaxScaler(), 
                    filter    = filt_model,      
                    reduction = red_model,       
                    clf       = mod_model        
                )
                
                try
                    # 2. CREAMOS LA MACHINE (Vital para que MLJ funcione)
                    mach = machine(pipe, X, y) 

                    # 3. EVALUAMOS
                    # Usamos acceleration=Serial() para evitar choques con MLJFlux (Redes Neuronales)
                    evaluation = evaluate!(
                        mach, 
                        resampling = folds, 
                        measures = measures, 
                        verbosity = 0,
                        acceleration = CPUThreads()
                    )
                    
                    # Extraemos resultados
                    acc_per_fold = evaluation.measurement[1]
                    f1_val       = evaluation.measurement[2]
                    b_acc_val    = evaluation.measurement[3]
                    
                    acc_mean = mean(acc_per_fold)
                    # Convertimos lista a string seguro para CSV (usando ; como separador)
                    acc_str = replace(string(acc_per_fold), "," => ";")
                    
                    println("Acc: $(round(acc_mean, digits=4)) | F1: $(round(f1_val, digits=4)) | B_acc: $(round(b_acc_val, digits=4))")
                    
                    # Guardamos
                    push!(results_df, (filt_name, red_name, mod_name, acc_str, acc_mean, f1_val, b_acc_val), promote=true)
                    CSV.write(output_file, results_df)
                    
                catch e
                    println("ERROR en $filt_name + $red_name + $mod_name:")
                    # Mostramos el error real para debug
                    showerror(stdout, e)
                    println("")
                    
                    # Guardamos el error en el CSV para no perder la fila
                    push!(results_df, (filt_name, red_name, mod_name, "ERROR", NaN, NaN, NaN))
                    CSV.write(output_file, results_df)
                end
                
                # Limpiamos memoria
                GC.gc() 
            end
        end
    end
    
    println("\n=== Experimento Finalizado ===")
    return results_df
end

run_experiment (generic function with 1 method)

In [ ]:
# Ejecutar y guardar
df_resultados_basicos = run_experiment(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_modelos_basicos.csv"
)

In [3]:
# ==============================================================================
# CONFIGURACIÓN DE LOS EXPERIMENTOS DE ENSEMBLE
# ==============================================================================

# 1. Configuración de Filtros (Solo probaremos sin filtrar de momento)
dic_filtros_basico = Dict(
    "Sin_Filtrado" => nothing
)

# 2. Configuración de Reducciones (PCA 95% y Sin Reducción)
dic_reducciones_ensemble = Dict(
    "Sin_Reduccion" => IdentityTransformer(), # O 'nothing' si prefieres
    "PCA_95"        => PCA(variance_ratio=0.95)
)

# 3. Definición de Modelos Base para los Ensembles
# ----------------------------------------------------------------
# Base para Bagging: KNN
knn_base = KNNClassifier(K=5)

# Base para AdaBoost: SVM Lineal 
# Usamos SGDClassifier con loss="hinge" que ES un SVM lineal, 
# pero compatible con el AdaBoost de ScikitLearn.
svm_base = SKSGDClassifier(
    loss         = "hinge",   # Hinge loss = comportamiento de SVM
    penalty      = "l2",      # Regularización estándar
    alpha        = 0.0001,    
    random_state = SEED       # Para reproducibilidad
)

# 4. Diccionario de Modelos de Ensemble
dic_modelos_ensemble = Dict(
    # --- Bagging (KNN) ---
    "Bagging_KNN_10" => EnsembleModel(
        model = knn_base,
        n     = 10
    ),
    "Bagging_KNN_50" => EnsembleModel(
        model = knn_base,
        n     = 50
    ),

    # --- AdaBoost (SVM Lineal) ---
    "AdaBoost_SVM" => AdaBoostClassifier(
        estimator = svm_base,
        n_estimators   = 5,
        algorithm      = "SAMME" # Obligatorio para SVM/Hinge loss
    ),

    # --- EvoTrees (Gradient Boosting) ---
    "EvoTree_50" => EvoTreeClassifier(
        nrounds = 50, 
        eta     = 0.2
    ),
    "EvoTree_100" => EvoTreeClassifier(
        nrounds = 100, 
        eta     = 0.2
    )
)

Dict{String, Probabilistic} with 5 entries:
  "EvoTree_50"     => EvoTreeClassifier(loss = mlogloss, …)
  "AdaBoost_SVM"   => AdaBoostClassifier(estimator = SGDClassifier(loss = hinge…
  "Bagging_KNN_10" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "Bagging_KNN_50" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "EvoTree_100"    => EvoTreeClassifier(loss = mlogloss, …)

In [ ]:
run_experiment(
    dic_filtros_basico, 
    dic_reducciones_ensemble, 
    dic_modelos_ensemble, 
    "resultados_ensembles_bagging.csv" # Guardamos en un archivo nuevo
)

Archivo de checkpoint encontrado: resultados_ensembles_bagging.csv
1 experimentos completados previamente.

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_10]
Acc: 0.894 | F1: 0.8941 | B_acc: 0.894

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_50]
Acc: 0.8931 | F1: 0.8933 | B_acc: 0.8932

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_100]
Acc: 0.8893 | F1: 0.8882 | B_acc: 0.8888

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_50]
Acc: 0.877 | F1: 0.8758 | B_acc: 0.8762

Evaluando: [Sin_Filtrado] + [Sin_Reduccion] + [AdaBoost_SVM]


┌ Error: Problem fitting the machine machine(:clf, …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695
┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704
┌ Error: Problem fitting machine(:clf, …)
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:767
┌ Error: Problem fitting the machine machine(PersonalizedPipeline(scaler = MyMinMaxScaler(), …), …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695


ERROR en Sin_Filtrado + Sin_Reduccion + AdaBoost_SVM:
TaskFailedException

┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704




    nested task error: Python: InvalidParameterError: The 'estimator' parameter of AdaBoostClassifier must be an object implementing 'fit' and 'predict' or None. Got Julia:
    SGDClassifier(
      loss = "hinge", 
      penalty = "l2", 
      alpha = 0.0001, 
      l1_ratio = 0.15, 
      fit_intercept = true, 
      max_iter = 1000, 
      tol = 0.001, 
      shuffle = true, 
      verbose = 0, 
      epsilon = 0.1, 
      n_jobs = nothing, 
      random_state = 104, 
      learning_rate = "optimal", 
      eta0 = 0.0, 
      power_t = 0.5, 
      early_stopping = false, 
      validation_fraction = 0.1, 
      n_iter_no_change = 5, 
      class_weight = nothing, 
      warm_start = false, 
      average = false) instead.
    Python stacktrace:
     [1] validate_parameter_constraints
       @ sklearn.utils._param_validation C:\Users\Pc\.julia\environments\v1.11\.CondaPkg\.pixi\envs\default\Lib\site-packages\sklearn\utils\_param_validation.py:95
     [2] _validate_params
       @ sk